# Google Pay-Inspired Expense Sharing

This beginner project shares travel expenses between **Alice, Bob, and Carol**. It uses a CSV file, calculates fair balances, creates a settlement plan, and shows two charts.

## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Project Settings and Friends

This example intentionally uses only three friends to keep the project easy to understand.

In [ ]:
friends = ["Alice", "Bob", "Carol"]
csv_file = "sample_expenses.csv"
output_folder = "generated"

os.makedirs(output_folder, exist_ok=True)

balances = {"Alice": 0.0, "Bob": 0.0, "Carol": 0.0}
fair_share = {"Alice": 0.0, "Bob": 0.0, "Carol": 0.0}
unpaid_bills = []

print("Friends:", ", ".join(friends))

## 3. Read the CSV Data

In [ ]:
expenses = pd.read_csv(csv_file)
expenses

## 4. Calculate Fair Shares and Balances

For each bill, the payer gets credit for the total amount. Each participant then pays their fair share. A blank `split_weights` value means equal split. A value such as `1|2|1` means 25%, 50%, and 25%.

In [ ]:
for _, row in expenses.iterrows():
    payer = str(row["payer"]).strip()
    amount = float(row["amount"])
    status = str(row["status"]).strip().lower()
    participants = [name.strip() for name in str(row["participants"]).split("|")]

    # An unpaid bill is shown as a warning, but is not used in settlement.
    if status == "unpaid":
        unpaid_bills.append(row)
        continue

    # A refund is treated as a negative expense.
    if status == "refund":
        amount = -amount

    # Skip a row if it has a name outside the three friends.
    if payer not in friends or any(person not in friends for person in participants):
        print("Skipped invalid row:", row["expense_id"])
        continue

    # Read custom weights, or use equal weights if the cell is blank.
    if pd.isna(row["split_weights"]) or str(row["split_weights"]).strip() == "":
        weights = [1] * len(participants)
    else:
        weights = [float(number) for number in str(row["split_weights"]).split("|")]

    if len(weights) != len(participants) or min(weights) <= 0:
        print("Skipped row because split weights are incorrect:", row["expense_id"])
        continue

    balances[payer] = balances[payer] + amount

    total_weight = sum(weights)
    for position in range(len(participants)):
        person = participants[position]
        share = amount * weights[position] / total_weight
        balances[person] = balances[person] - share
        fair_share[person] = fair_share[person] + share

## 5. Round the Money Values

Division can create a one-paise difference. This cell rounds values and keeps the total balance equal to zero.

In [ ]:
for friend in friends:
    balances[friend] = round(balances[friend], 2)
    fair_share[friend] = round(fair_share[friend], 2)

rounding_difference = round(sum(balances.values()), 2)
if rounding_difference != 0:
    biggest_balance = max(balances, key=lambda person: abs(balances[person]))
    balances[biggest_balance] = round(
        balances[biggest_balance] - rounding_difference, 2
    )

balances

## 6. Create the Settlement Plan

In [ ]:
debtors = []
creditors = []

for friend in friends:
    if balances[friend] < 0:
        debtors.append([friend, -balances[friend]])
    elif balances[friend] > 0:
        creditors.append([friend, balances[friend]])

payments = []
debtor_number = 0
creditor_number = 0

while debtor_number < len(debtors) and creditor_number < len(creditors):
    debtor = debtors[debtor_number]
    creditor = creditors[creditor_number]
    payment_amount = round(min(debtor[1], creditor[1]), 2)

    payments.append({"from": debtor[0], "to": creditor[0], "amount": payment_amount})

    debtor[1] = round(debtor[1] - payment_amount, 2)
    creditor[1] = round(creditor[1] - payment_amount, 2)

    if debtor[1] == 0:
        debtor_number += 1
    if creditor[1] == 0:
        creditor_number += 1

pd.DataFrame(payments)

## 7. Show the Result

In [ ]:
print("Final balances:")
for friend in friends:
    if balances[friend] > 0:
        print(friend, "should receive Rs.", balances[friend])
    elif balances[friend] < 0:
        print(friend, "owes Rs.", abs(balances[friend]))
    else:
        print(friend, "is settled")

print("\nSettlement plan:")
for payment in payments:
    print(payment["from"], "pays", payment["to"], "Rs.", payment["amount"])

if unpaid_bills:
    print("\nUnpaid bill warning:")
    for bill in unpaid_bills:
        print("-", bill["description"], "(Rs.", bill["amount"], ")")

## 8. Create Charts

The first chart shows group spending by category. The second chart shows the fair expense share for each friend.

In [ ]:
analysis_data = expenses[expenses["status"].str.lower() != "unpaid"].copy()
analysis_data["final_amount"] = np.where(
    analysis_data["status"].str.lower() == "refund",
    -analysis_data["amount"],
    analysis_data["amount"],
)

spending_by_category = analysis_data.groupby("category")["final_amount"].sum()
spending_by_category.plot(kind="bar", color="#4285F4", edgecolor="black")
plt.title("Net Spending by Category")
plt.xlabel("Category")
plt.ylabel("Amount (Rs.)")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "spending_by_category.png"), dpi=160)
plt.show()

In [ ]:
fair_share_data = pd.Series(fair_share).sort_values(ascending=False)
fair_share_data.plot(kind="bar", color="#34A853", edgecolor="black")
plt.title("Fair Expense Share by Friend")
plt.xlabel("Friend")
plt.ylabel("Amount (Rs.)")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "fair_share_by_friend.png"), dpi=160)
plt.show()

## 9. Save Result Files

In [ ]:
balance_data = pd.DataFrame({
    "friend": friends,
    "balance": [balances[friend] for friend in friends],
    "fair_share": [fair_share[friend] for friend in friends],
})

payment_data = pd.DataFrame(payments)
summary_data = pd.DataFrame({
    "metric": ["Net group expense", "Average settled bill", "Unpaid bills", "Settlement payments"],
    "value": [
        round(analysis_data["final_amount"].sum(), 2),
        round(np.mean(analysis_data["final_amount"].abs()), 2),
        len(unpaid_bills),
        len(payments),
    ],
})

expenses.to_csv(os.path.join(output_folder, "expense_data_used.csv"), index=False)
balance_data.to_csv(os.path.join(output_folder, "user_balances.csv"), index=False)
payment_data.to_csv(os.path.join(output_folder, "settlement_plan.csv"), index=False)
summary_data.to_csv(os.path.join(output_folder, "project_summary.csv"), index=False)

print("Saved charts and result CSV files in the generated folder.")
summary_data

## Key points

- Alice, Bob, and Carol are fixed sample users.
- A positive balance means the friend should receive money; a negative balance means the friend needs to pay.
- The unpaid snack is intentionally excluded because no person has paid it yet.
- The notebook creates charts and result CSV files in the `generated` folder.